# Sequence Modeling Phase 2: Mastering LSTM & GRU

Welcome to the deep dive into **Long Short-Term Memory (LSTM)** and **Gated Recurrent Units (GRU)**. In this notebook, we don't just look at equations—we solve the "Memory Crisis" of Deep Learning.

---

## 1. The Linguistic Motivation: Why RNNs Fail

Imagine a model reading this sentence:
> "The **girl**, who was wearing a beautiful dress and sitting near the window in the crowded room, picked up **her** book."

To predict the word **"her"**, the model must remember the gender of the subject (**"girl"**) across 15+ words. 
*   **Vanilla RNN:** Multiplies the gradient by the same weight matrix $W$ at every word. If $W=0.9$, after 15 words, the signal is $0.9^{15} \approx 0.2$. The gender is "washed out."
*   **LSTM:** Creates a dedicated "Highway" to carry that gender fact directly to the end of the sentence.

---

## 2. The Architecture: The Memory Highway

The core innovation of LSTM is the separation of memory into two states:
1.  **Cell State ($C_t$):** The **Long-Term Memory**. It flows like a highway with very little interaction, preserving information for hundreds of steps.
2.  **Hidden State ($h_t$):** The **Short-Term Context**. It is what the network "thinks" about the current word, used for making predictions.

![LSTM Internal Circuit](assets/lstm_circuit.gif)

---

## 3. Deep Dive into the Gates (Selecting, Inserting, Deleting)

LSTMs use **Gates** to regulate the flow. Each gate is a small neural network with a `sigmoid` activation ($0$ to $1$).

### A. The Forget Gate ($f_t$): The Eraser
**Usage:** When a period `.` appears in a text, we should forget the current sentence's context to prepare for the next.
**Intuition:** It decides which part of the old memory ($C_{t-1}$) is still relevant.
**Math:** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
*   If $f_t = 0$: "Delete this memory."
*   If $f_t = 1$: "Keep this memory perfectly."

### B. The Input Gate ($i_t$) & Candidate ($\tilde{C}_t$): The Highlighter
**Usage:** In the sentence "The girl...", the input gate highlights the fact "**girl**" and the candidate memory stores the gender "**female**".
**Intuition:** It decides which **new** information is worth adding to our long-term vault.
**Math:** 
*   $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ (The "How much" valve)
*   $\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$ (The "What" information)

### C. The Update Path: Constant Error Carousel
This is where the magic happens. Instead of multiplication, we use **Addition**:
$$ C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t $$
By using addition, the gradient can flow backward through the highway without shrinking. This is called the **Constant Error Carousel**.

### D. The Output Gate ($o_t$): The Filtered View
**Usage:** Even if we remember a person's name in the cell state, we only need to "output" it when the sentence specifically needs a subject.
**Intuition:** It decides which parts of our long-term memory should be used for the current hidden state $h_t$.
**Math:** $h_t = o_t \odot \tanh(C_t)$

---

## 4. How LSTM Solves Vanishing Gradients (The Proof)

In a standard RNN, the gradient $\frac{\partial h_T}{\partial h_1}$ is a product of weights: $\prod W$.
In an LSTM, the gradient $\frac{\partial C_T}{\partial C_1}$ follows the additive path. If the network learns to keep the **Forget Gate $f_t \approx 1$**, then:
$$ \frac{\partial C_t}{\partial C_{t-1}} \approx 1 $$
The gradient signal survives! The network **chooses** to keep its own gradient alive for important facts.

![Vanishing Solution](assets/vanishing_fix.gif)

---

## 5. GRU: The Efficient Evolution

The **Gated Recurrent Unit (GRU)** is a simplified version (Cho et al., 2014) that merges the Forget and Input gates into a single **Update Gate ($z_t$)**.

**Key Differences:**
1.  **Unified State:** No separate Cell State.
2.  **Reset Gate ($r_t$):** Decides how much of the past to ignore *before* calculating the update.
3.  **Speed:** ~33% faster to train due to fewer parameters.

![GRU Premium Logic](assets/gru_premium.gif)

---

## 6. Senior-Level Interview Prep

**Q1: Why is the separation of Cell State and Hidden State important?**
**Answer:** Conflict resolution. If you only have one state (like in RNN), it has to simultaneously serve as **long-term memory** and **immediate output context**. This causes a "clutter" problem. LSTMs decouple these, allowing the Cell State to store a fact quietly for 100 steps while the Hidden State focuses on the current local word-to-word transition.

**Q2: When would you use a Peephole LSTM?**
**Answer:** When the network needs **precise temporal timing**. For example, in music composition or high-frequency sensor data, the gates need to know *exactly* what's in the memory vault (Cell State) to decide when to release information, rather than just guessing based on the last output.

**Q3: Why initialize forget gate bias to a large value (1.0 or 2.0)?**
**Answer:** To avoid the "Initial Amnesia" problem. At the start of training, we want the network to **remember everything** by default. As training progresses, it will learn what to forget. If we start with 0, it forgets everything and never learns long-term patterns.


In [ ]:
import torch
import torch.nn as nn

# --- THE GOLD STANDARD: Manual LSTM Cell Logic ---
class ManualLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # We combine 4 gates (Forget, Input, Output, Candidate) into 1 matrix for speed
        self.W = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim)

    def forward(self, x, states):
        h_prev, c_prev = states
        combined = torch.cat((x, h_prev), dim=1)
        
        # 1. Compute all gate activations at once
        gates = self.W(combined) # Result is [batch, 4*hidden]
        f, i, o, c_tilde = gates.chunk(4, dim=1)
        
        # 2. Apply activations
        f = torch.sigmoid(f + 1.0) # Bias trick for long-term memory!
        i = torch.sigmoid(i)
        o = torch.sigmoid(o)
        c_tilde = torch.tanh(c_tilde)
        
        # 3. The Central Memory Highway (Cell State)
        # Multiplication for forgetting, Addition for remembering
        c_next = f * c_prev + i * c_tilde
        
        # 4. Filtered Output (Hidden State)
        h_next = o * torch.tanh(c_next)
        
        return h_next, c_next

# Validation
model = ManualLSTMCell(10, 20)
x_t = torch.randn(1, 10) # 1 word vector
h0, c0 = torch.zeros(1, 20), torch.zeros(1, 20)
h1, c1 = model(x_t, (h0, c0))

print(f"Propagated Cell Memory (Highway): {c1.abs().mean():.4f}")
print(f"Propagated Hidden State (Output): {h1.abs().mean():.4f}")
print("\nNote: Notice how c_next is a clean ADDITION of info, unlike Vanila RNN!")
